# Aegis: build, train and benchmark on Kaggle

End-to-end run on one dataset: functions → leak-free split → Joern code property graphs → CodeBERT node features → HGT model → PR-AUC / F1, compared with Semgrep and CodeQL on the same held-out functions.

Run it once per dataset (`DATASET` below): `bigvul` (C/C++, real-world), `juliet-java` and `juliet-cpp` (NIST Juliet Test Suite v1.3, synthetic).

**Before running (notebook sidebar → Settings):**
1. **Accelerator:** GPU (T4 or P100).
2. **Internet:** on (pip, Java, Joern, CodeQL, Semgrep rules, CodeBERT and the Juliet download).
3. **Input (BigVul only):** add a Kaggle dataset containing `MSR_data_cleaned.csv`. Juliet is downloaded from NIST automatically.
4. **If the GitHub repo is private:** add a Kaggle secret named `GITHUB_TOKEN` (Add-ons → Secrets) holding a read-only token for the repo.

**Resuming after a session ends:** graph building, training and the baseline scans all resume where they stopped, but only if their output directories survive. `/kaggle/working` is kept when you *Save Version*; to continue in a new session, add the previous version's output as an input and run the *Restore previous output* cell.

In [ ]:
DATASET = "bigvul"      # "bigvul", "juliet-java" or "juliet-cpp"
SAMPLE_SIZE = 2000      # functions to use; None for the whole dataset
BATCH_SIZE = 500        # functions parsed per Joern invocation
EPOCHS = 30
SEED = 42

REPO_URL = "github.com/Hikunokyojin/aegis.git"
JOERN_VERSION = "v4.0.597"
WORK = "/kaggle/working"
TAG = f"{DATASET}_{SAMPLE_SIZE or 'full'}"
GRAPHS_DIR = f"{WORK}/graphs_{TAG}"
RUN_DIR = f"{WORK}/runs/{TAG}"

In [ ]:
# Get the code: uses the GITHUB_TOKEN secret if present (private repo), otherwise a plain clone.
import os, subprocess

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass

url = f"https://{token}@{REPO_URL}" if token else f"https://{REPO_URL}"
if not os.path.exists(f"{WORK}/aegis"):
    subprocess.run(["git", "clone", "--depth", "1", url, f"{WORK}/aegis"], check=True)
print(subprocess.run(["git", "-C", f"{WORK}/aegis", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

In [ ]:
# Install the project with the exact dependency versions it was developed and tested with, plus Semgrep.
!pip install -q -e /kaggle/working/aegis semgrep
# Kaggle's preinstalled torchvision is built for its own torch version and crashes on import
# with ours ("operator torchvision::nms does not exist"); Aegis doesn't use it.
!pip uninstall -q -y torchvision

import torch, torch_geometric, transformers
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
print("torch_geometric", torch_geometric.__version__, "| transformers", transformers.__version__)
# If CUDA shows False here with the GPU accelerator on, the pinned torch wheel doesn't match
# Kaggle's GPU driver: reinstall with `pip install -q --no-deps -e /kaggle/working/aegis`
# followed by `pip install -q torch-geometric tree-sitter tree-sitter-cpp==0.23.4 tree-sitter-java==0.23.5 scikit-learn`
# to keep Kaggle's preinstalled torch instead.

In [ ]:
# Java 21 + Joern (same versions as the local setup the pipeline was verified against).
!apt-get -qq update && apt-get -qq install -y openjdk-21-jre-headless > /dev/null
!wget -q https://github.com/joernio/joern/releases/download/{JOERN_VERSION}/joern-cli-linux-x86_64.zip -O /tmp/joern.zip
!unzip -q -o /tmp/joern.zip -d /tmp/joern && rm /tmp/joern.zip

import glob
joern_dir = os.path.dirname(glob.glob("/tmp/joern/**/joern-parse", recursive=True)[0])
os.environ["PATH"] = joern_dir + ":" + os.environ["PATH"]
!java -version 2>&1 | head -1
!joern-parse --help | head -2

In [ ]:
# CodeQL CLI bundle (CLI + standard query packs), the second baseline. Semgrep was installed with pip above.
if not os.path.exists("/tmp/codeql/codeql"):
    !wget -q https://github.com/github/codeql-action/releases/latest/download/codeql-bundle-linux64.tar.gz -O /tmp/codeql.tgz
    !tar -xzf /tmp/codeql.tgz -C /tmp && rm /tmp/codeql.tgz
os.environ["PATH"] = "/tmp/codeql:" + os.environ["PATH"]
!codeql version | head -1
!semgrep --version

In [ ]:
# Restore previous output (only when continuing an earlier run): copies an earlier version's
# graphs_*/ and runs/ from /kaggle/input so the steps below resume instead of starting over.
import shutil
for previous in glob.glob("/kaggle/input/*/graphs_*") + glob.glob("/kaggle/input/*/runs"):
    target = f"{WORK}/{os.path.basename(previous)}"
    if not os.path.exists(target):
        shutil.copytree(previous, target)
        print("restored", previous, "->", target)

In [ ]:
# Locate the dataset: BigVul from the notebook inputs, Juliet downloaded from NIST's SARD.
JULIET_URLS = {
    "juliet-java": "https://samate.nist.gov/SARD/downloads/test-suites/2017-10-01-juliet-test-suite-for-java-v1-3.zip",
    "juliet-cpp": "https://samate.nist.gov/SARD/downloads/test-suites/2017-10-01-juliet-test-suite-for-c-cplusplus-v1-3.zip",
}
if DATASET == "bigvul":
    csv_matches = glob.glob("/kaggle/input/**/MSR_data_cleaned.csv", recursive=True)
    assert csv_matches, "Add the BigVul dataset (MSR_data_cleaned.csv) as a notebook input first."
    INPUT = csv_matches[0]
else:
    INPUT = f"/tmp/{DATASET}"
    if not os.path.exists(INPUT):
        !wget -q "{JULIET_URLS[DATASET]}" -O /tmp/juliet.zip
        !unzip -q -o /tmp/juliet.zip -d "{INPUT}" && rm /tmp/juliet.zip
print("dataset input:", INPUT)

In [ ]:
# Build training graphs. Resumable: re-running skips functions whose graph is already saved.
sample_arg = f"--sample-size {SAMPLE_SIZE}" if SAMPLE_SIZE else ""
!aegis-build-training-graphs --dataset {DATASET} --input "{INPUT}" --output-dir "{GRAPHS_DIR}" \
    --batch-size {BATCH_SIZE} --seed {SEED} {sample_arg} 2>&1 | grep -viE "warn|edges skipped"

In [ ]:
# How many vulnerable functions each split actually got (a split with none can't be scored).
import json
print(open(f"{GRAPHS_DIR}/split.json").read())
for split in ("train", "val", "test"):
    graphs = glob.glob(f"{GRAPHS_DIR}/{split}/graphs/*.pt")
    labels = [torch.load(g, weights_only=False)["label"] for g in graphs]
    print(f"{split:5s} {len(labels):6d} graphs, {sum(labels):5d} vulnerable")

In [ ]:
# Train. Resumable: re-running continues from the last finished epoch.
!aegis-train --data-dir "{GRAPHS_DIR}" --run-dir "{RUN_DIR}" --epochs {EPOCHS} 2>&1 | grep -viE "warn"

In [ ]:
# Benchmark: Aegis (best checkpoint by validation PR-AUC, threshold tuned on validation) vs
# Semgrep and CodeQL, all on the same held-out test functions. Writes results.json / results.md
# to RUN_DIR/evaluation; baseline scans are cached there, so a re-run doesn't rescan.
!aegis-evaluate --data-dir "{GRAPHS_DIR}" --run-dir "{RUN_DIR}" 2>&1 | grep -viE "warn"

In [ ]:
# Try the trained model on code it has never seen: the repo's own example files.
!aegis-scan --checkpoint "{RUN_DIR}/best.pt" /kaggle/working/aegis/tests/fixtures 2>&1 | grep -viE "warn"